[<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>](https://colab.research.google.com/github/ADRIANVM117/quant-research-notes/blob/main/volatility_target/01_rebalancing_policies/03_economic_statistic_evaluation.ipynb)

Abra la notebook en Google Colab y seleccione **Run all**. Los historiales certificados de P0, P1 y P2 se cargan automáticamente desde este repositorio; no se requieren claves de API, Google Drive ni ejecutar notebooks anteriores.

# Volatility Targeting | Evaluación estadística y económica

## Protocolo de investigación

### Pregunta de investigación

En las etapas anteriores construimos un estimador EWMA de volatilidad y un motor causal de rebalanceo para tres políticas de exposición dinámica a SPY.

Ahora queremos determinar si las bandas de tolerancia generan una mejora económica justificable frente al rebalanceo diario y frente a alternativas estáticas.

La pregunta central es:

> ¿Pueden las bandas de rebalanceo reducir los costos de implementación del volatility targeting sin deteriorar materialmente el control del riesgo ni comprometer el crecimiento geométrico del capital?

Esta investigación distingue tres dimensiones:

- **Eficiencia operativa:** frecuencia de negociación, turnover y costos de implementación.
- **Utilidad económica:** crecimiento del patrimonio y resultados netos después de costos.
- **Control del riesgo:** volatilidad realizada, desviación respecto al objetivo y pérdidas experimentadas.

Una reducción en el número de operaciones no demuestra, por sí sola, superioridad económica. Asimismo, obtener un CAGR mayor no implica necesariamente una política de inversión superior.

Nuestro propósito es estudiar conjuntamente estas dimensiones.

### Hipótesis de investigación

Evaluaremos la siguiente hipótesis:

> Introducir bandas de tolerancia puede disminuir la fricción operativa del volatility targeting y preservar un control de riesgo económicamente aceptable frente al rebalanceo diario.

Las bandas de 5 y 10 puntos porcentuales son parámetros previamente establecidos para esta investigación. No representan valores óptimos ni serán seleccionadas retrospectivamente según su rentabilidad.

Las diferencias observadas deberán distinguirse de la incertidumbre estadística y de posibles condiciones históricas favorables.

### Políticas y benchmarks

Compararemos cinco alternativas:

| Identificador | Política | Descripción |
|---|---|---|
| P0 | Daily | Volatility targeting con rebalanceo diario hacia el peso objetivo. |
| P1 | Banda 5 pp | Rebalanceo cuando la desviación absoluta supera 5 puntos porcentuales. |
| P2 | Banda 10 pp | Rebalanceo cuando la desviación absoluta supera 10 puntos porcentuales. |
| B1 | Buy & Hold SPY | Inversión inicial del capital disponible en SPY, sin rebalanceos posteriores. |
| B2 | Asignación estática 60/40 | Asignación inicial de 60% a SPY y 40% a efectivo, permitiendo que los pesos evolucionen sin rebalanceos. |

P0 constituye la referencia principal para evaluar el valor incremental de las bandas.

B1 permite comparar las políticas dinámicas con la exposición completa al mercado.

B2 proporciona una referencia sencilla de exposición inicialmente moderada, sin aplicar pronósticos de volatilidad.

Estas alternativas no tienen necesariamente el mismo nivel de riesgo. Por ello, una comparación de rendimientos absolutos será insuficiente para determinar su utilidad relativa.

### Contrato de comparabilidad

Para evitar ventajas artificiales, todas las políticas y benchmarks deberán evaluarse bajo condiciones consistentes:

- Mismo periodo y calendario de sesiones.
- Mismo capital inicial: 200,000 MXN, convertido a USD bajo la convención documentada.
- Misma moneda contable para comparar resultados.
- Mismo tratamiento de dividendos y efectivo.
- Costos de implementación explícitos y aplicables según las operaciones de cada alternativa.
- Ausencia de información futura en decisiones de inversión.
- Trazabilidad entre las series de rendimientos y los estados de cartera que las originan.

Los resultados de las políticas P0, P1 y P2 procederán del motor desarrollado en la notebook 02. Los nuevos benchmarks deberán respetar las mismas convenciones económicas.

### Alcance de la evidencia

La muestra histórica ya fue utilizada durante el desarrollo y la exploración de las políticas.

Por tanto, los resultados de esta notebook constituirán inicialmente evidencia retrospectiva, no confirmación independiente fuera de muestra.

Las futuras pruebas de estabilidad e incertidumbre servirán para evaluar la solidez de los hallazgos, sin eliminar las limitaciones derivadas de haber inspeccionado previamente el periodo histórico.

**Objetivo del bloque:** establecer referencias comparables y un protocolo de evaluación antes de calcular resultados económicos o realizar inferencia estadística.

## Bloque 05 — Construcción de benchmarks

Este bloque construirá y verificará los insumos necesarios para comparar P0, P1, P2, B1 y B2 bajo condiciones económicas consistentes.

| Sección | Contenido |
| ------- | ------------------------------------------------------ |
| 5.1 | Insumos reproducibles y validación de P0/P1/P2 |
| 5.2 | Benchmark B1 — Buy & Hold SPY |
| 5.3 | Benchmark B2 — Asignación estática 60/40 |
| 5.4 | Validación de comparabilidad de las cinco alternativas |

### 5.1. Insumos reproducibles y validación de historiales

Las políticas P0, P1 y P2 se generaron con el motor causal de la notebook 02 y se preparan como historiales Parquet certificados. Cada archivo conserva el estado diario de cartera, las operaciones, costos, dividendos y efectivo. El manifiesto documenta sus parámetros, convenciones y hashes SHA-256.

La siguiente celda busca primero los artefactos versionados en el repositorio local. Si no están disponibles, los descarga desde GitHub. Antes de cargarlos verifica su integridad y que los tres historiales compartan el mismo contrato de evaluación.

In [ ]:
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlretrieve
import hashlib
import importlib.util
import json
import math
import subprocess
import sys

if importlib.util.find_spec('polars') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'polars==1.44.2'])

import polars as pl

ARTIFACT_FILENAMES = {
    'P0 — Daily': 'policy_p0.parquet',
    'P1 — Banda 5 pp': 'policy_p1.parquet',
    'P2 — Banda 10 pp': 'policy_p2.parquet',
}
MANIFEST_FILENAME = 'manifest.json'
REMOTE_ARTIFACTS_URL = (
    'https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/main/'
    'volatility_target/artifacts/evaluation_inputs'
)
ESSENTIAL_COLUMNS = {
    'policy', 'session_date', 'signal_date', 'target_weight',
    'effective_weight_close', 'shares', 'cash', 'dividend_receivable',
    'cash_interest', 'dividend_net', 'dividend_payment', 'open_price',
    'close_price', 'executed_shares', 'trade_executed', 'trade_notional',
    'turnover_notional', 'commission', 'iva', 'total_cost',
    'portfolio_value_open', 'portfolio_value_close',
}

def find_local_artifact_dir() -> Path | None:
    required = [MANIFEST_FILENAME, *ARTIFACT_FILENAMES.values()]
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidates = (
            folder / 'volatility_target' / 'artifacts' / 'evaluation_inputs',
            folder / 'artifacts' / 'evaluation_inputs',
        )
        for candidate in candidates:
            if all((candidate / filename).is_file() for filename in required):
                return candidate
    return None

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1_048_576), b''):
            digest.update(chunk)
    return digest.hexdigest()

artifact_dir = find_local_artifact_dir()
if artifact_dir is None:
    artifact_dir = Path.cwd() / 'evaluation_inputs'
    artifact_dir.mkdir(parents=True, exist_ok=True)
    for filename in [MANIFEST_FILENAME, *ARTIFACT_FILENAMES.values()]:
        destination = artifact_dir / filename
        temporary = destination.with_suffix(destination.suffix + '.part')
        try:
            urlretrieve(f'{REMOTE_ARTIFACTS_URL}/{filename}', temporary)
            temporary.replace(destination)
        except URLError as error:
            raise RuntimeError(
                'No fue posible descargar los insumos de evaluación desde GitHub. '
                'Verifica que artifacts/evaluation_inputs esté publicado.'
            ) from error

manifest_path = artifact_dir / MANIFEST_FILENAME
try:
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
except (OSError, json.JSONDecodeError) as error:
    raise RuntimeError('manifest.json no se pudo leer como un manifiesto válido.') from error

if manifest.get('manifest_version') != 1:
    raise ValueError('La versión del manifiesto no es compatible.')
if manifest.get('artifact_type') != 'volatility_target_policy_histories':
    raise ValueError('El manifiesto no corresponde a historiales de volatility targeting.')

parameters = manifest.get('strategy_parameters', {})
expected_parameters = {
    'ewma_lambda': 0.94,
    'target_volatility_annualized': 0.12,
    'max_exposure': 1.0,
}
for name, expected in expected_parameters.items():
    if not math.isclose(float(parameters.get(name, float('nan'))), expected):
        raise ValueError(f'El parámetro {name} no coincide con la convención documentada.')
expected_bands = {'P0 — Daily': None, 'P1 — Banda 5 pp': 0.05, 'P2 — Banda 10 pp': 0.10}
if parameters.get('bands') != expected_bands:
    raise ValueError('Las bandas del manifiesto no coinciden con P0, P1 y P2.')

conventions = manifest.get('economic_conventions', {})
required_conventions = {
    'decision_and_execution', 'slippage_bps', 'dividend_withholding',
    'cash_interest', 'stock_commission_per_share_usd',
}
if required_conventions.difference(conventions):
    raise ValueError('El manifiesto no documenta todas las convenciones económicas.')

certificates = manifest.get('policies', {})
policy_histories = {}
reference_dates = None
summary_rows = []

for policy, filename in ARTIFACT_FILENAMES.items():
    certificate = certificates.get(policy)
    if not certificate:
        raise ValueError(f'El manifiesto no contiene el certificado de {policy}.')
    path = artifact_dir / filename
    if not path.is_file():
        raise FileNotFoundError(f'Falta el historial requerido: {path.name}')
    if sha256_file(path) != certificate.get('sha256'):
        raise ValueError(f'Falló la verificación SHA-256 de {path.name}.')

    history = pl.read_parquet(path)
    missing = ESSENTIAL_COLUMNS.difference(history.columns)
    if missing:
        raise ValueError(f'{policy}: faltan columnas esenciales: {sorted(missing)}')
    if history.is_empty():
        raise ValueError(f'{policy}: el historial está vacío.')
    nulls_in_essential_columns = history.select([
        pl.col(column).is_null().sum().alias(column) for column in ESSENTIAL_COLUMNS
    ])
    if sum(nulls_in_essential_columns.row(0)) != 0:
        raise ValueError(f'{policy}: existen valores faltantes en columnas esenciales.')
    if history['policy'].n_unique() != 1 or history['policy'][0] != policy:
        raise ValueError(f'{policy}: el identificador de política es inconsistente.')

    dates = history['session_date'].to_list()
    if dates != sorted(dates) or len(dates) != len(set(dates)):
        raise ValueError(f'{policy}: las fechas deben ser únicas y ordenadas.')
    if not (history['portfolio_value_close'] > 0).all():
        raise ValueError(f'{policy}: se detectó un patrimonio no positivo.')
    if history.height != certificate.get('sessions'):
        raise ValueError(f'{policy}: el número de sesiones no coincide con su certificado.')
    if str(dates[0]) != certificate.get('first_date') or str(dates[-1]) != certificate.get('last_date'):
        raise ValueError(f'{policy}: las fechas no coinciden con su certificado.')
    if not math.isclose(
        float(history['portfolio_value_open'][0]),
        float(certificate.get('initial_portfolio_value_usd')),
        rel_tol=0.0,
        abs_tol=1e-8,
    ):
        raise ValueError(f'{policy}: el patrimonio inicial no coincide con su certificado.')
    if not math.isclose(
        float(history['portfolio_value_close'][-1]),
        float(certificate.get('final_portfolio_value')),
        rel_tol=0.0,
        abs_tol=1e-8,
    ):
        raise ValueError(f'{policy}: el patrimonio final no coincide con el certificado de la notebook 02.')

    if reference_dates is None:
        reference_dates = dates
    elif dates != reference_dates:
        raise ValueError('P0, P1 y P2 no comparten el mismo calendario de sesiones.')
    policy_histories[policy] = history
    summary_rows.append({
        'policy': policy,
        'sessions': history.height,
        'first_date': str(dates[0]),
        'last_date': str(dates[-1]),
        'initial_capital_usd': float(manifest['initial_capital']['net_usd']),
        'final_portfolio_value': float(history['portfolio_value_close'][-1]),
    })

if (
    len(reference_dates) != manifest['period']['sessions']
    or str(reference_dates[0]) != manifest['period']['first_date']
    or str(reference_dates[-1]) != manifest['period']['last_date']
):
    raise ValueError('El calendario común no coincide con el periodo documentado.')

policy_p0 = policy_histories['P0 — Daily']
policy_p1 = policy_histories['P1 — Banda 5 pp']
policy_p2 = policy_histories['P2 — Banda 10 pp']
validation_summary = pl.DataFrame(summary_rows)
validation_summary